# Omo Forest NDVI Forecasting Project
Vegetation time-series forecasting using Sentinel-2 NDVI and an LSTM.

**Pipeline:** GEE setup -> NDVI time series extraction -> sequence building -> LSTM training -> evaluation -> visualization.

**Cost:** $0 (Google Earth Engine Community tier + free Colab runtime).

## Step 1: Setup — Earth Engine authentication and AOI

In [ ]:
!pip install geemap --quiet

import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Replace 'your-project-id' with your actual GEE Cloud Project ID
ee.Authenticate()
ee.Initialize(project='your-project-id')

print("Earth Engine initialized successfully.")


In [ ]:
# Area of Interest: Omo Forest Reserve, Ogun State, Nigeria
omo_forest = ee.Geometry.Rectangle([4.19, 6.35, 4.65, 7.05])

Map = geemap.Map(center=[6.7, 4.4], zoom=9)
Map.addLayer(omo_forest, {'color': 'red'}, 'Omo Forest AOI (rough)')
Map


## Step 2: Extract monthly NDVI time series

In [ ]:
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
        qa.bitwiseAnd(cirrus_bit_mask).eq(0)
    )
    return image.updateMask(mask).divide(10000)


def add_ndvi(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)


def get_monthly_ndvi(aoi, start_year=2018, end_year=2026, cloud_threshold=60):
    # cloud_threshold relaxed from 30 -> 60: Omo Forest is tropical rainforest with
    # near-constant cloud cover, so a strict 30% threshold rejected most months
    # outright. 60 keeps more real observations while the per-pixel cloud mask
    # (mask_s2_clouds) still scrubs the actual cloudy pixels out of each image.
    records = []
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            start = ee.Date.fromYMD(year, month, 1)
            end = start.advance(1, 'month')

            collection = (
                ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                .filterBounds(aoi)
                .filterDate(start, end)
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold))
                .map(mask_s2_clouds)
                .map(add_ndvi)
            )

            count = collection.size().getInfo()
            if count == 0:
                continue

            # Median composite is more robust to residual cloud/shadow noise
            # than mean, especially with a looser cloud threshold.
            monthly_composite = collection.select('NDVI').median()

            median_ndvi = monthly_composite.reduceRegion(
                reducer=ee.Reducer.median(),
                geometry=aoi,
                scale=100,
                maxPixels=1e9
            ).get('NDVI')

            try:
                ndvi_value = median_ndvi.getInfo()
            except Exception:
                ndvi_value = None

            if ndvi_value is not None:
                records.append({
                    'year': year,
                    'month': month,
                    'date': f'{year}-{month:02d}-01',
                    'ndvi': ndvi_value,
                    'image_count': count
                })
                print(f'{year}-{month:02d}: NDVI = {ndvi_value:.4f} ({count} images)')

    return pd.DataFrame(records)


In [ ]:
# Full 2018-2026 range with the relaxed cloud filter. This will take a while
# (roughly 8-9 years x 12 months of GEE calls) but should recover far more
# real observations than the earlier 2023-2026 / 30% threshold run did.
ndvi_df = get_monthly_ndvi(omo_forest, start_year=2018, end_year=2026, cloud_threshold=60)
ndvi_df.head(20)


In [ ]:
ndvi_df.to_csv('omo_ndvi_monthly.csv', index=False)

n_possible_months = (2026 - 2018 + 1) * 12
print(f"Real observed months: {len(ndvi_df)} out of {n_possible_months} possible.")
print(f"Saved {len(ndvi_df)} monthly records to omo_ndvi_monthly.csv")


## Step 3: Explore and clean the time series

In [ ]:
ndvi_df['date'] = pd.to_datetime(ndvi_df['date'])
ndvi_df = ndvi_df.sort_values('date').reset_index(drop=True)

plt.figure(figsize=(12, 4))
plt.plot(ndvi_df['date'], ndvi_df['ndvi'], marker='o')
plt.title('Monthly Mean NDVI — Omo Forest Reserve')
plt.xlabel('Date')
plt.ylabel('NDVI')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(ndvi_df['ndvi'].describe())


In [ ]:
# Fill gaps (months with no clear imagery) via interpolation so the LSTM sequence
# is continuous -- but explicitly flag which points are real vs interpolated,
# since that distinction matters for honestly reporting model performance.
ndvi_df = ndvi_df.set_index('date').asfreq('MS')
ndvi_df['is_observed'] = ndvi_df['ndvi'].notna()
ndvi_df['ndvi'] = ndvi_df['ndvi'].interpolate(method='linear')
ndvi_df = ndvi_df.reset_index()

n_observed = ndvi_df['is_observed'].sum()
n_total = len(ndvi_df)
print(f"Real observed months: {n_observed} / {n_total} ({n_observed / n_total:.0%})")
print(f"Interpolated (filled) months: {n_total - n_observed}")

ndvi_df[['date', 'ndvi', 'is_observed']].head(10)


## Step 4: Build sequences for the LSTM
Use a sliding window of past months to predict the next month's NDVI.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

WINDOW_SIZE = 6  # number of past months used to predict the next one
N_FEATURES = 3   # NDVI (scaled) + month_sin + month_cos

# Seasonality features: encode calendar month as a point on a circle so the
# model can learn recurring seasonal patterns (e.g. dry-season dips) instead
# of only reacting to changes after they've already happened.
ndvi_df['month_sin'] = np.sin(2 * np.pi * ndvi_df['date'].dt.month / 12)
ndvi_df['month_cos'] = np.cos(2 * np.pi * ndvi_df['date'].dt.month / 12)

ndvi_values = ndvi_df['ndvi'].values.reshape(-1, 1)
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_ndvi = scaler.fit_transform(ndvi_values)

# month_sin/cos are already bounded in [-1, 1], no scaling needed
seasonal_features = ndvi_df[['month_sin', 'month_cos']].values

feature_matrix = np.hstack([scaled_ndvi, seasonal_features])  # shape (n, 3)


def make_sequences(data, window_size, target_col=0):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size, :])
        y.append(data[i + window_size, target_col])
    return np.array(X), np.array(y)


X, y = make_sequences(feature_matrix, WINDOW_SIZE)
# X shape is already (samples, timesteps, features) -- no reshape needed now

print(f"X shape: {X.shape}, y shape: {y.shape}")


In [ ]:
# Chronological train/test split (do NOT shuffle time series data)
split_idx = int(len(X) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")


## Step 5: Build and train the LSTM

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    LSTM(32, activation='tanh', input_shape=(WINDOW_SIZE, N_FEATURES), return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()


In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=8,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Training History')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Step 6: Evaluate and visualize the forecast

In [ ]:
y_pred = model.predict(X_test)

# Inverse-transform back to real NDVI scale
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))
y_pred_actual = scaler.inverse_transform(y_pred)

from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(y_test_actual, y_pred_actual)
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))

print(f"Test MAE: {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")

# How much of the test set is real observation vs interpolated filler --
# this context matters when judging how meaningful the error metrics are.
test_observed_flags = ndvi_df['is_observed'].values[-len(y_test):]
print(f"Test set real observations: {test_observed_flags.sum()} / {len(test_observed_flags)}")


In [ ]:
test_dates = ndvi_df['date'].values[-len(y_test):]

plt.figure(figsize=(12, 5))
plt.plot(test_dates, y_test_actual, label='Actual NDVI', marker='o')
plt.plot(test_dates, y_pred_actual, label='Predicted NDVI', marker='x')
plt.title('LSTM NDVI Forecast vs Actual — Omo Forest Reserve')
plt.xlabel('Date')
plt.ylabel('NDVI')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Step 7 (Optional): Forecast forward beyond available data

In [ ]:
def forecast_future(model, last_window, scaler, last_date, n_steps=6):
    predictions = []
    current_window = last_window.copy()  # shape (WINDOW_SIZE, N_FEATURES)

    for step in range(1, n_steps + 1):
        pred = model.predict(current_window.reshape(1, WINDOW_SIZE, N_FEATURES), verbose=0)
        pred_ndvi_scaled = pred[0, 0]
        predictions.append(pred_ndvi_scaled)

        # Compute the real calendar month for this forecast step, so the
        # seasonality features stay meaningful instead of being reused/stale.
        future_date = last_date + pd.DateOffset(months=step)
        month_sin = np.sin(2 * np.pi * future_date.month / 12)
        month_cos = np.cos(2 * np.pi * future_date.month / 12)

        new_row = np.array([pred_ndvi_scaled, month_sin, month_cos])
        current_window = np.vstack([current_window[1:], new_row])

    predictions = np.array(predictions).reshape(-1, 1)
    return scaler.inverse_transform(predictions)


last_window = feature_matrix[-WINDOW_SIZE:, :]
last_date = ndvi_df['date'].iloc[-1]
future_ndvi = forecast_future(model, last_window, scaler, last_date, n_steps=6)

print("Forecasted NDVI for next 6 months:")
for i, val in enumerate(future_ndvi.flatten(), start=1):
    print(f"  Month +{i}: {val:.4f}")


## Step 8: Spatial NDVI Visualization
The time series shows *when* vegetation health changed, but not *where*. This step renders the actual NDVI raster as a color-coded map so degraded vs healthy zones within Omo Forest are visible directly.

In [ ]:
def get_ndvi_composite_for_month(aoi, year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')

    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
        .map(mask_s2_clouds)
        .map(add_ndvi)
    )
    return collection.select('NDVI').median()


# Pick two months to compare -- e.g. a healthy-looking month vs a degraded one,
# based on the values already printed in Step 2's output.
ndvi_healthy = get_ndvi_composite_for_month(omo_forest, 2024, 5)   # example: high NDVI month
ndvi_degraded = get_ndvi_composite_for_month(omo_forest, 2025, 1)  # example: low NDVI month

vis_params = {'min': 0.2, 'max': 0.8, 'palette': ['brown', 'yellow', 'green']}

Map = geemap.Map(center=[6.7, 4.4], zoom=10)
Map.addLayer(ndvi_healthy, vis_params, 'NDVI - Healthier Month (2024-05)')
Map.addLayer(ndvi_degraded, vis_params, 'NDVI - Degraded Month (2025-01)')
Map.add_colorbar(vis_params, label='NDVI')
Map


In [ ]:
# Export a static snapshot for the writeup/README (no interactive controls needed there)
geemap.ee_export_image(
    ndvi_healthy.clip(omo_forest),
    filename='ndvi_healthy_month.tif',
    scale=100,
    region=omo_forest
)
print("Exported ndvi_healthy_month.tif -- open in QGIS or convert to PNG for the README.")


## Optional Enrichment: FAOSTAT Nigeria yield correlation
If you want to relate NDVI trends to real agricultural yield, download free Nigeria crop yield data from [FAOSTAT](https://www.fao.org/faostat/en/#data/QCL) and merge by year for a coarse correlation check. This is a stretch goal, not required for the core forecasting deliverable.

## Step 9: Export polished charts for README / logbook
Publication-style versions of the key charts, saved as PNG files ready to embed in a GitHub README or paste into the logbook writeup.

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')

# --- Chart 1: Full NDVI history, observed vs interpolated ---
fig, ax = plt.subplots(figsize=(12, 5))
observed = ndvi_df[ndvi_df['is_observed']]
interpolated = ndvi_df[~ndvi_df['is_observed']]

ax.plot(ndvi_df['date'], ndvi_df['ndvi'], color='seagreen', linewidth=1.5, zorder=1)
ax.scatter(observed['date'], observed['ndvi'], color='seagreen', s=35, label='Observed', zorder=2)
ax.scatter(interpolated['date'], interpolated['ndvi'], color='lightgray', s=35,
           label='Interpolated', zorder=2, edgecolors='gray')

ax.set_title('Omo Forest Reserve — Monthly NDVI (2018-2026)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('NDVI')
ax.legend()
plt.tight_layout()
plt.savefig('chart_ndvi_history.png', dpi=200)
plt.show()
print("Saved chart_ndvi_history.png")


In [ ]:
# --- Chart 2: Forecast vs Actual, polished ---
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(test_dates, y_test_actual, color='seagreen', marker='o', linewidth=2,
        label='Actual NDVI', markersize=7)
ax.plot(test_dates, y_pred_actual, color='darkorange', marker='x', linewidth=2,
        linestyle='--', label='LSTM Forecast', markersize=8)

ax.set_title('LSTM NDVI Forecast vs Actual — Omo Forest Reserve', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('NDVI')
ax.legend()
ax.text(
    0.02, 0.02, f'MAE: {mae:.3f}  |  RMSE: {rmse:.3f}',
    transform=ax.transAxes, fontsize=10,
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
)
plt.tight_layout()
plt.savefig('chart_forecast_vs_actual.png', dpi=200)
plt.show()
print("Saved chart_forecast_vs_actual.png")


In [ ]:
# --- Chart 3: Training history, polished ---
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(history.history['loss'], color='steelblue', linewidth=2, label='Train Loss')
ax.plot(history.history['val_loss'], color='darkorange', linewidth=2, label='Val Loss')
ax.set_title('LSTM Training History', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.legend()
plt.tight_layout()
plt.savefig('chart_training_history.png', dpi=200)
plt.show()
print("Saved chart_training_history.png")
print("\nAll charts saved. Download them from the Colab file browser (folder icon, left sidebar).")
